In [1]:
import pandas as pd

In [2]:
import os
import sys

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

In [10]:
from src.config import settings
from src.embeddings.e5_model import E5Model
from src.models.vacancy import Vacancy
from src.pipeline.recommendation_pipeline import RecommendationPipeline
from src.preprocess.e5_small_formatter import E5Formatter
from src.vector_store.qdrant_store import QdrantStore


In [4]:
vacancies_df = pd.read_csv('../data/processed/cleaned_vacancies.csv')
vacancies_df.head(2)

,vacancy_id,title,author_name,description,city,salary_min,salary_max,requirements,conditions,metro,currency,experience_min,experience_max,tags,remote_type,time_type,author_id
0,49313809,Golang Developer (Кипр),Space307,Мы в Space307 разрабатываем международную торг...,Санкт-Петербург,251322.0,NaN,"Программист, разработчик",Условия обсуждаются на собеседовании,NaN,RUB,3,6.0,"docker, golang, redis, английский язык, kafka",OFFICE,FULL,4563d26a-8438-4934-99de-1363a2150c06
1,48813842,Е-mail маркетолог,Монополия,С 2015 года наш IT блок меняет рынок автотранс...,Санкт-Петербург,60900.0,NaN,Менеджер по маркетингу и рекламе,Условия обсуждаются на собеседовании,NaN,RUB,1,3.0,"грамотность, написание текстов, грамотная речь...",OFFICE,FULL,1b0aba1d-1e3f-4757-acb3-5d869c5658e4


In [7]:
model = E5Model(model_name=settings.embedding_model, batch_size=64)

store = QdrantStore(
    host=settings.qdrant_host,
    port=settings.qdrant_port,
    collection_name=settings.collection_name,
    vector_size=model.dimension,
)

formatter = E5Formatter()

pipeline = RecommendationPipeline(
    name='test_e5',
    formatter=formatter,
    embedding_model=model,
    vector_store=store,
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 9541.88it/s]


In [12]:
vacancies_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 47325 entries, 0 to 47324
Data columns (total 17 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   vacancy_id      47325 non-null  int64  
 1   title           47325 non-null  str    
 2   author_name     47325 non-null  str    
 3   description     47325 non-null  str    
 4   city            47325 non-null  str    
 5   salary_min      15079 non-null  float64
 6   salary_max      10037 non-null  float64
 7   requirements    47286 non-null  str    
 8   conditions      47325 non-null  str    
 9   metro           0 non-null      float64
 10  currency        47325 non-null  str    
 11  experience_min  47325 non-null  int64  
 12  experience_max  45759 non-null  float64
 13  tags            47325 non-null  str    
 14  remote_type     47325 non-null  str    
 15  time_type       47325 non-null  str    
 16  author_id       47325 non-null  str    
dtypes: float64(4), int64(2), str(11)
memory us

In [ ]:
from tqdm import tqdm


def upload_dataframe_to_qdrant(
    df: pd.DataFrame, 
    pipeline, 
    batch_size: int = 500
):
    """
    Пошагово конвертирует DataFrame в объекты Vacancy и загружает в Qdrant батчами.
    """
    total_rows = len(df)
    print(f"Начинаем загрузку {total_rows} вакансий (размер батча: {batch_size})...")

    for start_idx in tqdm(range(0, total_rows, batch_size), desc="Загрузка в Qdrant"):
        chunk_df = df.iloc[start_idx : start_idx + batch_size]

        raw_records = chunk_df.to_dict(orient="records")
        clean_records = [
            {k: (None if pd.isna(v) else v) for k, v in row.items()}
            for row in raw_records
        ]

        vacancies = [Vacancy(**row) for row in clean_records]

        pipeline.add_vacancy(vacancies)

    print("\nВся база успешно загружена в Qdrant!")

In [17]:
upload_dataframe_to_qdrant(
    df=vacancies_df, 
    pipeline=pipeline, 
    batch_size=500
)

Начинаем загрузку 47325 вакансий (размер батча: 500)...


Загрузка в Qdrant: 100%|██████████| 95/95 [10:56<00:00,  6.91s/it]


Вся база успешно загружена в Qdrant!


In [20]:
res = pipeline.search_by_id(19503786)
for v in res.items:
    print(v.id)
    print(v.score)
    print(v.metadata)
    print('-' * 20)

34323750
0.9782254
{'vacancy_id': 34323750, 'title': 'Backend\xa0разработчик (Java)', 'author_name': 'НИЦ СПб ЭТУ (Научно-инженерный центр Санкт-Петербургского электротехнического университета)', 'author_id': '79544054-fe95-46e3-b672-64626bacabc5', 'tags': '[]', 'city': 'Санкт-Петербург', 'metro': None, 'currency': 'RUB', 'remote_type': 'OFFICE', 'time_type': 'FULL', 'salary_min': None, 'salary_max': None, 'experience_min': 1, 'experience_max': 3}
--------------------
34890535
0.97412544
{'vacancy_id': 34890535, 'title': 'Разработчик (Java)', 'author_name': 'НИЦ СПб ЭТУ (Научно-инженерный центр Санкт-Петербургского электротехнического университета)', 'author_id': '79544054-fe95-46e3-b672-64626bacabc5', 'tags': 'apache maven, linux, git, java, ооп, javascript', 'city': 'Санкт-Петербург', 'metro': None, 'currency': 'RUB', 'remote_type': 'OFFICE', 'time_type': 'FULL', 'salary_min': None, 'salary_max': None, 'experience_min': 1, 'experience_max': 3}
--------------------
39759368
0.9566516
